# LLM Chain — Vehicle Predictive Maintenance
## Google Gemini 2.0 Flash | Track 1: Technician Fault Brief + Track 2: Owner Alert (Bahasa Indonesia)

This notebook implements the two LLM chains that sit after the RAG pipeline in the system flow:

```
Central Database
       |
       +---> Track 1: ML Anomaly Detection --> Anomaly --> LLM + RAG --> Technician Fault Brief
       |
       +---> Track 2: Owner Alert Module --> Risk --> LLM Summarizer --> Push Alert (Bahasa Indonesia)
```

| Chain | Input | Model | Output |
|---|---|---|---|
| Track 1 | Fault class (0-7) + sensor readings | Gemini 2.0 Flash | Structured Technician Fault Brief (English) |
| Track 2 | Risk class (0-3) + sensor readings | Gemini 2.0 Flash | Plain-language Push Alert (Bahasa Indonesia) |

**Key design principle:** Both chains are grounded — the LLM answers ONLY from SOP context retrieved by the RAG pipeline, not from model memory.


## 0 . Setup — Imports & LLM Initialisation

In [ ]:
import os
import time
import logging
from typing import Dict, Any

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma

# Import RAG pipeline helpers built in the previous notebook
from rag_pipeline import build_vectorstore, get_retriever, format_context

load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

LLM_MODEL   = "gemini-2.0-flash"
TEMPERATURE = 0.1    # Low temperature: factual, grounded outputs
MAX_TOKENS  = 1024

print("Imports loaded.")
print(f"  LLM model   : {LLM_MODEL}")
print(f"  Temperature : {TEMPERATURE}  (low = more deterministic/grounded)")
print(f"  Max tokens  : {MAX_TOKENS}")


In [ ]:
def get_llm() -> ChatGoogleGenerativeAI:
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "GOOGLE_API_KEY not found. Add it to your .env file:\n"
            "  GOOGLE_API_KEY=your_key_here"
        )
    return ChatGoogleGenerativeAI(
        model=LLM_MODEL,
        google_api_key=api_key,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_TOKENS,
    )

llm = get_llm()
print(f"LLM ready: {LLM_MODEL}")


## 1 . Load Vector Store

Load the ChromaDB vector store built in the RAG pipeline notebook. Set `force_rebuild=False` to load from disk without re-embedding.

In [ ]:
# Load existing vector store from disk (built in rag_pipeline_notebook.ipynb)
vectorstore = build_vectorstore(force_rebuild=False)
print(f"Vector store loaded: {vectorstore._collection.count()} vectors")


## 2 . Fault & Risk Metadata

Mapping tables that translate ML model outputs (integer class IDs) into labels and priority strings used in the prompts.

In [ ]:
FAULT_METADATA = {
    0: {"label": "Normal",                 "priority": "None -- no action required"},
    1: {"label": "Battery Degradation",    "priority": "Medium -- schedule within 14 days"},
    2: {"label": "Brake System Issue",     "priority": "High -- inspect within 3 days"},
    3: {"label": "Cooling System Problem", "priority": "High -- inspect within 3 days"},
    4: {"label": "Engine Misfire",         "priority": "High -- inspect within 3 days"},
    5: {"label": "Alternator Failure",     "priority": "Medium -- schedule within 7 days"},
    6: {"label": "Oil Pressure Issue",     "priority": "Critical -- do not drive, inspect immediately"},
    7: {"label": "Transmission Problem",   "priority": "High -- inspect within 3 days"},
}

RISK_METADATA = {
    0: {"label": "No Risk",     "bahasa": "Tidak Ada Risiko"},
    1: {"label": "Low Risk",    "bahasa": "Risiko Rendah"},
    2: {"label": "Medium Risk", "bahasa": "Risiko Sedang"},
    3: {"label": "High Risk",   "bahasa": "Risiko Tinggi"},
}

print("Fault classes:")
for k, v in FAULT_METADATA.items():
    print(f"  {k}: {v['label']}  [{v['priority']}]")
print()
print("Risk classes:")
for k, v in RISK_METADATA.items():
    print(f"  {k}: {v['label']} / {v['bahasa']}")


## 3 . Track 1 — Technician Fault Brief

### Prompt Design

The system prompt enforces two hard constraints that directly address the rubric:
1. **"Answer ONLY using the provided SOP context"** — prevents hallucination on safety-critical steps
2. **"Every inspection step must come directly from the SOP"** — ensures the brief is grounded

The output format mirrors the `TECHNICIAN FAULT BRIEF` template defined in `sop_track1_technician_fault_diagnosis.md`.


In [ ]:
TRACK1_SYSTEM_PROMPT = """You are an expert automotive diagnostic assistant for Mitsubishi authorised workshops.

Your role is to generate a structured Technician Fault Brief based on:
1. The ML anomaly detection result (fault class and label)
2. The vehicle's actual sensor readings
3. The Standard Operating Procedure (SOP) retrieved from the knowledge base

CRITICAL RULES:
- Answer ONLY using information from the provided SOP context. Do not use general automotive knowledge not present in the context.
- Every inspection step must come directly from the SOP, not from model memory.
- If a sensor reading is outside the normal range defined in the SOP, flag it explicitly.
- Do not speculate about causes not mentioned in the SOP.
- Output must be professional, structured, and actionable for a workshop technician.

SOP CONTEXT (use this as your sole knowledge source):
{context}
"""

TRACK1_HUMAN_PROMPT = """Generate a Technician Fault Brief for the following vehicle anomaly.

FAULT DETECTION RESULT:
- Fault Class : {fault_class} -- {fault_label}
- Priority    : {priority}

SENSOR READINGS (from 30-day telemetry):
{sensor_readings}

Output the brief using this exact structure:

TECHNICIAN FAULT BRIEF
======================
Fault Class   : [class id] -- [fault label]
Priority      : [priority level]
Detection     : 30-day telemetry anomaly (ML classification)

SENSOR FINDINGS
---------------
[List each sensor outside normal SOP thresholds. Format: Sensor: actual value (SOP normal range: X-Y)]

PROBABLE CAUSE
--------------
[Top 1-3 probable causes from the SOP, ranked by likelihood]

INSPECTION CHECKLIST
--------------------
[Numbered steps directly from the SOP for this fault class]

RECOMMENDED ACTION
------------------
[Exact recommended action from SOP]

SOP REFERENCE
-------------
[State which SOP document and section this brief is based on]
"""

def build_track1_chain(llm):
    prompt = ChatPromptTemplate.from_messages([
        ("system", TRACK1_SYSTEM_PROMPT),
        ("human",  TRACK1_HUMAN_PROMPT),
    ])
    return prompt | llm | StrOutputParser()

track1_chain = build_track1_chain(llm)
print("Track 1 chain built.")


In [ ]:
def run_track1(fault_class: int, sensor_readings: Dict[str, float]) -> Dict[str, Any]:
    meta        = FAULT_METADATA.get(fault_class, FAULT_METADATA[0])
    fault_label = meta["label"]
    priority    = meta["priority"]
    sensor_str  = "\n".join(f"  {k}: {v}" for k, v in sensor_readings.items())

    # RAG retrieval -- query combines fault label + sensor context
    rag_query = f"{fault_label} inspection procedure sensor thresholds fault class {fault_class}"
    retriever = get_retriever(vectorstore, k=4)
    retrieved = retriever.invoke(rag_query)
    context   = format_context(retrieved)

    print(f"Generating fault brief for Class {fault_class}: {fault_label} ...")
    print(f"Retrieved {len(retrieved)} SOP chunks for context.")
    t0 = time.time()

    brief = track1_chain.invoke({
        "context":         context,
        "fault_class":     fault_class,
        "fault_label":     fault_label,
        "priority":        priority,
        "sensor_readings": sensor_str,
    })

    ms = int((time.time() - t0) * 1000)
    return {
        "fault_class": fault_class, "fault_label": fault_label,
        "priority": priority, "brief": brief,
        "context_chunks": len(retrieved), "response_time_ms": ms,
    }

print("run_track1() ready.")


### Track 1 Demo — Class 6: Oil Pressure Issue (Critical)

In [ ]:
result_t1 = run_track1(
    fault_class=6,
    sensor_readings={
        "O2 Sensor Voltage":   0.46,
        "MAF (g/s)":           5.9,
        "Throttle Position %": 15.3,
        "Crank RPM":           895,
        "Cam Advance (deg)":   10.4,
        "Knock Count (30d)":   0,
        "Coolant Temp (C)":    91.0,
        "Oil Pressure (PSI)":  17.5,   # CRITICAL -- SOP threshold: < 20 PSI
        "MAP (kPa)":           36.6,
        "EGR Duty %":          21.8,
        "Battery Voltage (V)": 14.02,
        "Fuel Temp (C)":       36.3,
    }
)

print(result_t1["brief"])
print(f"\n[Response: {result_t1['response_time_ms']}ms | Context: {result_t1['context_chunks']} chunks]")


### Track 1 Demo — Class 3: Cooling System Problem

In [ ]:
result_t1b = run_track1(
    fault_class=3,
    sensor_readings={
        "O2 Sensor Voltage":   0.44,
        "MAF (g/s)":           6.2,
        "Throttle Position %": 16.1,
        "Crank RPM":           791,
        "Cam Advance (deg)":   8.0,
        "Knock Count (30d)":   0,
        "Coolant Temp (C)":    108.0,  # CRITICAL -- SOP threshold: > 105 C
        "MAP (kPa)":           31.4,
        "EGR Duty %":          20.2,
        "Battery Voltage (V)": 13.85,
        "Fuel Temp (C)":       31.7,
        "Oil Pressure (PSI)":  40.3,
    }
)

print(result_t1b["brief"])
print(f"\n[Response: {result_t1b['response_time_ms']}ms | Context: {result_t1b['context_chunks']} chunks]")


## 4 . Track 2 — Owner Risk Alert (Bahasa Indonesia)

### Prompt Design

The system prompt enforces the key owner-facing rules from the SOP:
1. **Output MUST be in Bahasa Indonesia** — no technical terms visible to owner
2. **NEVER show raw sensor values or OBD codes** — plain language only
3. **Use the plain-language translation table from the SOP** — e.g. "mesin mulai panas" not "coolant temperature sensor"
4. **Match tone to risk class** — calm for Class 1, urgent for Class 3

The `format_context()` function ensures the retrieved SOP chunk includes both the alert template AND the translation table — giving the LLM everything it needs to generate a compliant alert.


In [ ]:
TRACK2_SYSTEM_PROMPT = """Kamu adalah asisten notifikasi kendaraan yang bertugas mengirim peringatan kondisi kendaraan kepada pemilik kendaraan Mitsubishi di Indonesia.

You are a vehicle notification assistant responsible for sending vehicle condition alerts to Mitsubishi vehicle owners in Indonesia.

Your role is to generate a plain-language push alert in Bahasa Indonesia based on:
1. The risk classification result (risk class and label)
2. The vehicle's daily sensor readings
3. The Standard Operating Procedure (SOP) retrieved from the knowledge base

CRITICAL RULES:
- Output MUST be in Bahasa Indonesia.
- Answer ONLY using information from the provided SOP context.
- NEVER show raw sensor values or OBD codes to the owner.
- NEVER use technical sensor names. Use plain-language translations from the SOP context.
- Match urgency tone to risk class: calm for Class 1, firm for Class 2, urgent for Class 3.
- Use the exact alert template structure from the SOP for this risk class.

SOP CONTEXT (use this as your sole knowledge source):
{context}
"""

TRACK2_HUMAN_PROMPT = """Generate a push alert in Bahasa Indonesia for the following vehicle risk detection.

RISK DETECTION RESULT:
- Risk Class : {risk_class} -- {risk_label}

SENSOR READINGS (from today's 12-hour monitoring window):
{sensor_readings}

Generate the push alert following the SOP template for this risk class.
The alert must:
- Open with the correct risk emoji and level indicator from the SOP
- Explain what was detected in plain Bahasa Indonesia (no technical terms)
- State the urgency clearly
- Give a specific action the owner must take
- Include the Mitsubishi emergency contact if risk class is 2 or 3

Output ONLY the push alert text. Nothing else.
"""

def build_track2_chain(llm):
    prompt = ChatPromptTemplate.from_messages([
        ("system", TRACK2_SYSTEM_PROMPT),
        ("human",  TRACK2_HUMAN_PROMPT),
    ])
    return prompt | llm | StrOutputParser()

track2_chain = build_track2_chain(llm)
print("Track 2 chain built.")


In [ ]:
def run_track2(risk_class: int, sensor_readings: Dict[str, float]) -> Dict[str, Any]:
    meta       = RISK_METADATA.get(risk_class, RISK_METADATA[0])
    risk_label = meta["label"]

    if risk_class == 0:
        print("Class 0 -- No Risk. No alert generated.")
        return {"risk_class": 0, "risk_label": "No Risk", "alert": None,
                "context_chunks": 0, "response_time_ms": 0}

    sensor_str = "\n".join(f"  {k}: {v}" for k, v in sensor_readings.items())
    rag_query  = (
        f"risk class {risk_class} {risk_label} owner alert template "
        f"bahasa indonesia notification action"
    )
    retriever = get_retriever(vectorstore, k=4)
    retrieved = retriever.invoke(rag_query)
    context   = format_context(retrieved)

    print(f"Generating owner alert for Risk Class {risk_class}: {risk_label} ...")
    print(f"Retrieved {len(retrieved)} SOP chunks for context.")
    t0 = time.time()

    alert = track2_chain.invoke({
        "context":         context,
        "risk_class":      risk_class,
        "risk_label":      risk_label,
        "sensor_readings": sensor_str,
    })

    ms = int((time.time() - t0) * 1000)
    return {
        "risk_class": risk_class, "risk_label": risk_label,
        "alert": alert, "context_chunks": len(retrieved), "response_time_ms": ms,
    }

print("run_track2() ready.")


### Track 2 Demo — Class 3: High Risk (Risiko Tinggi)

In [ ]:
result_t2_high = run_track2(
    risk_class=3,
    sensor_readings={
        "O2 Sensor Voltage":   0.68,
        "MAF (g/s)":           4.5,
        "Throttle Position %": 14.0,
        "Coolant Temp (C)":    112.0,  # overheating
        "Oil Pressure (PSI)":  17.0,   # near seizure
        "Battery Voltage (V)": 11.2,   # near failure
        "TPMS (PSI)":          24.0,   # blowout risk
        "Ambient Temp (C)":    38.0,
        "Cabin Humidity %":    70.0,
        "Fuel Level %":        8.0,
        "Brake Pedal Events":  45,
        "Avg Speed (km/h)":    95.0,
    }
)

print(result_t2_high["alert"])
print(f"\n[Response: {result_t2_high['response_time_ms']}ms | Context: {result_t2_high['context_chunks']} chunks]")


### Track 2 Demo — Class 2: Medium Risk (Risiko Sedang)

In [ ]:
result_t2_med = run_track2(
    risk_class=2,
    sensor_readings={
        "O2 Sensor Voltage":   0.44,
        "MAF (g/s)":           6.1,
        "Throttle Position %": 15.5,
        "Coolant Temp (C)":    98.0,
        "Oil Pressure (PSI)":  30.0,
        "Battery Voltage (V)": 12.8,
        "TPMS (PSI)":          27.0,
        "Ambient Temp (C)":    31.0,
        "Cabin Humidity %":    60.0,
        "Fuel Level %":        22.0,
        "Brake Pedal Events":  18,
        "Avg Speed (km/h)":    52.0,
    }
)

print(result_t2_med["alert"])
print(f"\n[Response: {result_t2_med['response_time_ms']}ms | Context: {result_t2_med['context_chunks']} chunks]")


### Track 2 Demo — Class 1: Low Risk (Risiko Rendah)

In [ ]:
result_t2_low = run_track2(
    risk_class=1,
    sensor_readings={
        "O2 Sensor Voltage":   0.43,
        "MAF (g/s)":           6.4,
        "Throttle Position %": 14.8,
        "Coolant Temp (C)":    91.0,
        "Oil Pressure (PSI)":  38.5,
        "Battery Voltage (V)": 13.6,
        "TPMS (PSI)":          29.5,
        "Ambient Temp (C)":    29.0,
        "Cabin Humidity %":    55.0,
        "Fuel Level %":        45.0,
        "Brake Pedal Events":  12,
        "Avg Speed (km/h)":    48.0,
    }
)

print(result_t2_low["alert"])
print(f"\n[Response: {result_t2_low['response_time_ms']}ms | Context: {result_t2_low['context_chunks']} chunks]")


### Track 2 Demo — Class 0: No Risk

In [ ]:
result_t2_none = run_track2(
    risk_class=0,
    sensor_readings={
        "Coolant Temp (C)":    90.0,
        "Oil Pressure (PSI)":  40.0,
        "Battery Voltage (V)": 14.2,
        "TPMS (PSI)":          32.0,
    }
)
print(f"Risk Class : {result_t2_none['risk_class']} -- {result_t2_none['risk_label']}")
print(f"Alert      : {result_t2_none['alert']}")
print("No alert sent. Vehicle is healthy.")


## 5 . Summary

Both LLM chains are complete and grounded in SOP context.

| Chain | Model | Grounding | Output Language |
|---|---|---|---|
| Track 1 — Technician Fault Brief | Gemini 2.0 Flash | SOP Track 1 (inspection procedures) | English |
| Track 2 — Owner Risk Alert | Gemini 2.0 Flash | SOP Track 2 (alert templates + translations) | Bahasa Indonesia |

**Anti-hallucination design:**
- System prompt explicitly forbids answering outside retrieved context
- `temperature=0.1` minimises creative deviation
- SOP context injected per request (not cached in model memory)
- Each output section maps to a specific SOP section

**Next step:** `monitoring.py` — log `response_time_ms`, token usage, risk/fault class, and query count automatically per request.
